<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 · SEMANA 5 — SESIÓN 2 · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 3 — BM25 y evaluación de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Mejor ranking que TF-IDF, y cómo medirlo con métricas de IR</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.
- **Sin librerías de NLP/IR para resolver:** TF-IDF, BM25, las métricas y K-Means van **desde cero**. `scikit-learn` solo se permite donde se indique *verificación*.


## Objetivo

Dos partes. **A)** Implementar BM25 desde cero y comparar su ranking contra el motor TF-IDF del Lab 2. **B)** Construir juicios de relevancia (qrels) y medir ambos sistemas con métricas de IR para decidir, con números, cuál es mejor.


## 0 · Corpus procesado del Lab 1

In [5]:
import json, math, re, unicodedata
from collections import Counter
import spacy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')


with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)              # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


### Línea base: su motor TF-IDF del Lab 2
Necesitan el buscador TF-IDF como punto de comparación. Peguen sus funciones del Lab 2.

In [7]:
# TODO: peguen sus funciones del Lab 2: tf, idf, tfidf, coseno, preprocesar,
#       y construyan IDF, INDICE, vectorizar_consulta y buscar_tfidf(q, k=5).

RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')
RE_EMOJI = re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF]')

stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'sin'}   # palabras que NO quieren tratar como stopword (con criterio de dominio)
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto, quitar_acentos=True):
    """Devuelve el texto normalizado (aun como string, sin tokenizar)."""
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.lower()
    texto = RE_HTML.sub(' ', texto)
    texto = RE_URL.sub(' ', texto)
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def preprocesar(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.lower()
        if lemma in MIS_STOPWORDS:
            continue
        if len(lemma) <= 1 and not lemma.isdigit():
            continue
        tokens.append(lemma)
    return tokens

def tf(doc):
    """doc: lista de terminos -> dict termino:tf (frecuencia relativa)."""
    total = len(doc)
    if total == 0:
        return {}
    counts = Counter(doc)
    return {term: count / total for term, count in counts.items()}

def idf(corpus):
    """corpus: lista de docs (cada uno lista de terminos) -> dict termino:idf."""
    n_docs = len(corpus)
    df = Counter()
    for doc in corpus:
        for term in set(doc):  # set() asegura contar el doc una sola vez por término
            df[term] += 1
    return {term: math.log(n_docs / df_t) for term, df_t in df.items()}

def tfidf(doc, idf_):
    """-> dict termino:peso tf-idf para un documento."""
    tf_doc = tf(doc)
    return {term: tf_val * idf_.get(term, 0) for term, tf_val in tf_doc.items()}

# Construir el indice: un vector tf-idf (dict) por documento
IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

# Sanity check: el termino mas pesado de d04 (sequia/maiz) deberia ser tematico
import operator
top = sorted(INDICE[3].items(), key=operator.itemgetter(1), reverse=True)[:5]
print('Terminos top de', corpus[3]['id'], '->', top)

def vectorizar_consulta(texto):
    """texto libre -> vector tf-idf (dict) usando el IDF del corpus."""
    texto_proc = preprocesar(texto)
    return tfidf(texto_proc, IDF)

print(vectorizar_consulta('sequia en los cultivos'))

def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts). Maneja el caso de norma 0."""
    # Producto punto: solo recorremos las claves de v1 (las que no estan en v2 aportan 0)
    dot = sum(peso * v2.get(term, 0) for term, peso in v1.items())

    norma1 = math.sqrt(sum(peso ** 2 for peso in v1.values()))
    norma2 = math.sqrt(sum(peso ** 2 for peso in v2.values()))

    if norma1 == 0 or norma2 == 0:
        return 0.0

    return dot / (norma1 * norma2)

def buscar_tfidf(q, k=5):
    """Devuelve los k documentos mas relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(q)

    resultados = [
        (doc['id'], doc['titulo'], coseno(q, vector))   
        for doc, vector in zip(corpus, INDICE)
    ]

    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados[:k]

Terminos top de d04 -> [('gravemente', 0.16494108310095365), ('cultivo', 0.16494108310095365), ('maiz', 0.16494108310095365), ('frijol', 0.16494108310095365), ('fronterizo', 0.16494108310095365)]
{'sequia': 0.9729550745276566, 'cultivo': 1.3195286648076292}


---
## Parte A · BM25 desde cero

**A.1** IDF de BM25 (variante suavizada, nunca negativa) y la función `bm25`. Sigan la fórmula de la clase:
$$\text{score}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f\,(k_1+1)}{f+k_1\,(1-b+b\,|d|/\text{avgdl})}$$

In [8]:
avgdl = sum(len(doc) for doc in documentos) / len(documentos)

def idf_bm25(corpus):
    # TODO: ln(1 + (N - df + 0.5)/(df + 0.5))  por termino
    n_docs = len(corpus)
    df = Counter()
    for doc in corpus:
        for term in set(doc):
            df[term] += 1
    return {
        term: math.log(1 + (n_docs - df_t + 0.5) / (df_t + 0.5))
        for term, df_t in df.items()
    }


def bm25(doc, q, k1=1.5, b=0.75):
    # TODO: acumular el score termino por termino segun la formula
    counts = Counter(doc)
    dl = len(doc)
    score = 0.0
    for term in q:
        f = counts.get(term, 0)
        if f == 0:
            continue
        idf_t = IDF_BM25.get(term, 0)
        numerador = f * (k1 + 1)
        denominador = f + k1 * (1 - b + b * (dl / avgdl))
        score += idf_t * (numerador / denominador)
    return score

# Construir IDF de BM25 sobre todo el corpus (una sola vez)
IDF_BM25 = idf_bm25(documentos)

**A.2** Buscador BM25, análogo a `buscar_tfidf` pero ranqueando por score BM25.

In [9]:
def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    # TODO: preprocesar la consulta -> score bm25 contra cada doc -> top-k
    q = preprocesar(consulta)
    resultados = [
        (doc['id'], doc['titulo'], bm25(d, q, k1=k1, b=b))
        for doc, d in zip(corpus, documentos)
    ]
    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados[:k]

**A.3** Comparación lado a lado. Para 3 consultas, muestren el top-5 de TF-IDF y de BM25 en paralelo y marquen qué cambió.

In [14]:
# TODO: 3 consultas; imprimir top-5 de TF-IDF y BM25 lado a lado y senalar diferencias.
# 3 consultas; imprimir top-5 de TF-IDF y BM25 lado a lado y senalar diferencias.

consultas_prueba = [
    'sequia en los cultivos',
    'precio del cafe',
    'sequia y cultivos de maiz',
]

for consulta in consultas_prueba:
    print(f"Consulta: '{consulta}'")

    top_tfidf = buscar_tfidf(consulta, k=5)   # ajusta el nombre si la dejaste como buscar()
    top_bm25 = buscar_bm25(consulta, k=5)

    print(f"\n{'TF-IDF':<38} | {'BM25':<38}")
    print('-' * 80)
    for (id1, tit1, s1), (id2, tit2, s2) in zip(top_tfidf, top_bm25):
        izq = f"{id1} ({s1:.3f}) {tit1[:22]}"
        der = f"{id2} ({s2:.3f}) {tit2[:22]}"
        print(f"{izq:<38} | {der:<38}")
        
print("\nObservaciones:")
print("Consulta 1: " + consultas_prueba[0])
print("Aparecen los mismos documentos en el mismo orden, cambia unicamente el score, BM25 supera la unidad mientras que TF-IDF se mantiene por debajo de 1")
print("Consulta 2: " + consultas_prueba[1])
print("Aparecen los mismos documentos en el mismo orden, cambia unicamente el score, BM25 supera la unidad mientras que TF-IDF se mantiene por debajo de 1")
print("Consulta 3: " + consultas_prueba[2])
print("Aparecen los mismos documentos en el mismo orden, cambia unicamente el score, BM25 supera la unidad mientras que TF-IDF se mantiene por debajo de 1")


Consulta: 'sequia en los cultivos'

TF-IDF                                 | BM25                                  
--------------------------------------------------------------------------------
d04 (0.335) Sequia afecta cultivos     | d04 (4.143) Sequia afecta cultivos    
d02 (0.107) Crisis hidrica golpea      | d02 (1.674) Crisis hidrica golpea     
d01 (0.000) Lluvias provocan inund     | d01 (0.000) Lluvias provocan inund    
d03 (0.000) Cafe de Chiapas rompe      | d03 (0.000) Cafe de Chiapas rompe     
d05 (0.000) Turismo crece en el Ca     | d05 (0.000) Turismo crece en el Ca    
Consulta: 'precio del cafe'

TF-IDF                                 | BM25                                  
--------------------------------------------------------------------------------
d03 (0.334) Cafe de Chiapas rompe      | d03 (4.143) Cafe de Chiapas rompe     
d12 (0.118) Feria celebra el cafe      | d12 (1.813) Feria celebra el cafe     
d01 (0.000) Lluvias provocan inund     | d01 (0.000) 

**A.4** Barrido de parámetros. Observen cómo se mueve el ranking de una consulta al variar (k₁, b).

In [13]:
# TODO: barrer (k1, b) in {1.2, 2.0} x {0.0, 0.75, 1.0} para una consulta
#       e imprimir el top-3 en cada caso.
# barrer (k1, b) in {1.2, 2.0} x {0.0, 0.75, 1.0} para una consulta
#       e imprimir el top-3 en cada caso.

consulta_sweep = 'sequia en los cultivos'

for k1_val in [1.2, 2.0]:
    for b_val in [0.0, 0.75, 1.0]:
        print(f"\n--- k1={k1_val}, b={b_val} ---")
        top3 = buscar_bm25(consulta_sweep, k=3, k1=k1_val, b=b_val)
        for id_, titulo, score in top3:
            print(f"  {id_}  {score:.3f}  {titulo}")


--- k1=1.2, b=0.0 ---
  d04  4.094  Sequia afecta cultivos de maiz
  d02  1.792  Crisis hidrica golpea la region
  d01  0.000  Lluvias provocan inundaciones en Tuxtla

--- k1=1.2, b=0.75 ---
  d04  4.139  Sequia afecta cultivos de maiz
  d02  1.684  Crisis hidrica golpea la region
  d01  0.000  Lluvias provocan inundaciones en Tuxtla

--- k1=1.2, b=1.0 ---
  d04  4.153  Sequia afecta cultivos de maiz
  d02  1.651  Crisis hidrica golpea la region
  d01  0.000  Lluvias provocan inundaciones en Tuxtla

--- k1=2.0, b=0.0 ---
  d04  4.094  Sequia afecta cultivos de maiz
  d02  1.792  Crisis hidrica golpea la region
  d01  0.000  Lluvias provocan inundaciones en Tuxtla

--- k1=2.0, b=0.75 ---
  d04  4.148  Sequia afecta cultivos de maiz
  d02  1.662  Crisis hidrica golpea la region
  d01  0.000  Lluvias provocan inundaciones en Tuxtla

--- k1=2.0, b=1.0 ---
  d04  4.167  Sequia afecta cultivos de maiz
  d02  1.622  Crisis hidrica golpea la region
  d01  0.000  Lluvias provocan inundaciones 

_Describan cómo y por qué cambia el ranking al mover k₁ y b:_ 

---
## Parte B · Evaluación con métricas de IR

**B.1** Juicios de relevancia (qrels). Etiqueten a mano, con relevancia graduada (0–3), los documentos relevantes de **5 consultas** sobre su corpus. Completen el diccionario.

In [16]:
# relevancia graduada: 3 = muy relevante, 2 = relevante, 1 = marginal, ausente = 0

qrels = {
    'sequia y cultivos':  {'d04': 3, 'd02': 2},   # ejemplo dado

    'precio del cafe': {
        'd03': 3,  # tokens incluyen 'cafe', 'precio', 'alza', 'exportacion' -> tema central
        'd12': 2,  # 'cafe', 'cacao', 'venta', 'productor' -> mismo sector pero el foco es la feria, no el precio
        'd08': 1,  # 'produccion', 'cacao', 'mercado', 'premium' -> mismo rubro agroexportador, pero ni cafe ni precio
    },

    'turismo en chiapas': {
        'd05': 3,  # 'visitante', 'recorrido', 'atractivo', 'parque nacional' -> articulo central de turismo
        'd09': 3,  # 'destino', 'cultural', 'atraer', 'viajero' -> igualmente articulo central de turismo
        'd12': 1,  # 'feria', 'asistente', 'recorrer', 'stand' -> evento que atrae publico, pero no es nota de turismo
    },

    'escasez de agua': {
        'd02': 3,  # 'crisis', 'hidrico', 'desabasto', 'escasez' -> tema central exacto
        'd13': 2,  # 'agua', 'potable', 'falla', 'red' -> problema de suministro de agua, causa distinta (infraestructura, no sequia)
        'd04': 1,  # 'sequia' aparece, pero el foco del articulo es perdida de cultivos, no el agua en si
    },

    'tecnologia e innovacion en chiapas': {
        'd07': 3,  # 'inteligencia', 'artificial', 'laboratorio', 'aprendizaje', 'automatico' -> tema central
        'd14': 3,  # 'robotica', 'robotico', 'ingenieria', 'competencia', 'internacional' -> tema central
        'd10': 1,  # 'obra', 'carretera', 'mejorar' -> es infraestructura, no tecnologia, pero comparte el espiritu de "modernizacion"
    },
}
assert len(qrels) >= 5, 'Faltan consultas por etiquetar'

**B.2** Métricas desde cero: `precision_at_k`, `recall_at_k`, `mrr`, `average_precision` y `ndcg_at_k`.

In [17]:
def _rel(qid, doc): return qrels[qid].get(doc, 0)
def _relevantes(qid): return {d for d, g in qrels[qid].items() if g > 0}

def precision_at_k(ranking, qid, k=5):
    top_k = ranking[:k]
    relevantes_recuperados = sum(1 for doc_id, _, _ in top_k if _rel(qid, doc_id) > 0)
    return relevantes_recuperados / k

def recall_at_k(ranking, qid, k=5):
    total_relevantes = _relevantes(qid)
    if not total_relevantes:
        return 0.0
    top_k = ranking[:k]
    recuperados = {doc_id for doc_id, _, _ in top_k if _rel(qid, doc_id) > 0}
    return len(recuperados) / len(total_relevantes)

def mrr(ranking, qid):
    for i, (doc_id, _, _) in enumerate(ranking, start=1):
        if _rel(qid, doc_id) > 0:
            return 1 / i
    return 0.0

def average_precision(ranking, qid):
    total_relevantes = _relevantes(qid)
    if not total_relevantes:
        return 0.0
    aciertos = 0
    suma_precisiones = 0.0
    for i, (doc_id, _, _) in enumerate(ranking, start=1):
        if _rel(qid, doc_id) > 0:
            aciertos += 1
            suma_precisiones += aciertos / i
    return suma_precisiones / len(total_relevantes)

def ndcg_at_k(ranking, qid, k=5):
    def dcg(relevancias):
        # i empieza en 0 -> log2(i+2): la primera posicion usa log2(2)=1
        return sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevancias))

    top_k = ranking[:k]
    rels_obtenidos = [_rel(qid, doc_id) for doc_id, _, _ in top_k]
    dcg_val = dcg(rels_obtenidos)

    rels_ideales = sorted(qrels[qid].values(), reverse=True)[:k]
    idcg_val = dcg(rels_ideales)

    return dcg_val / idcg_val if idcg_val > 0 else 0.0

**B.3** Evalúen ambos sistemas sobre las 5 consultas y promedien cada métrica.

In [19]:
# TODO: funcion evaluar(buscar_fn) que promedie las 5 metricas sobre los qrels;
#       construyan la tabla comparativa TF-IDF vs BM25 con la columna de mejora.

def evaluar(buscar_fn, k=5):
    """Promedia P@k, R@k, MRR, AP y NDCG@k de buscar_fn sobre todas las consultas de qrels."""
    acumulado = {'P@5': 0.0, 'R@5': 0.0, 'MRR': 0.0, 'MAP': 0.0, 'NDCG@5': 0.0}
    n = len(qrels)

    for qid in qrels:
        ranking = buscar_fn(qid, k=k)
        acumulado['P@5']    += precision_at_k(ranking, qid, k)
        acumulado['R@5']    += recall_at_k(ranking, qid, k)
        acumulado['MRR']    += mrr(ranking, qid)
        acumulado['MAP']    += average_precision(ranking, qid)
        acumulado['NDCG@5'] += ndcg_at_k(ranking, qid, k)

    return {metrica: total / n for metrica, total in acumulado.items()}


def tabla_comparativa(buscar_a, nombre_a, buscar_b, nombre_b, k=5):
    res_a = evaluar(buscar_a, k)
    res_b = evaluar(buscar_b, k)

    print(f"{'Metrica':<10} {nombre_a:>10} {nombre_b:>10} {'Mejora':>10}")
    print('-' * 44)
    for metrica in res_a:
        v_a, v_b = res_a[metrica], res_b[metrica]
        if v_a > 0:
            mejora_str = f"{(v_b - v_a) / v_a * 100:+.1f}%"
        else:
            mejora_str = "n/a" if v_b == 0 else "+inf%"
        print(f"{metrica:<10} {v_a:>10.3f} {v_b:>10.3f} {mejora_str:>10}")

    return res_a, res_b


resultados_tfidf, resultados_bm25 = tabla_comparativa(
    buscar_tfidf, 'TF-IDF', buscar_bm25, 'BM25'
)

Metrica        TF-IDF       BM25     Mejora
--------------------------------------------
P@5             0.360      0.360      +0.0%
R@5             0.667      0.667      +0.0%
MRR             0.750      0.867     +15.6%
MAP             0.557      0.596      +7.0%
NDCG@5          0.680      0.732      +7.6%


**B.4** ¿Qué sistema y qué (k₁, b) eligen para producción, y **con qué métrica** lo justifican?

_Su decisión:_ 

Elijo BM25.
En la tabla anterior se peude observar que P@5 y R@5 son iguales tanto con TF-IDF y en BM25, puesto que no evalúan orden, mientras que MRR, MAP y NDCG@5 si tienen una diferencia, a favor de BM25,teniendo una mejora de **15.6%, 7.0% y 7.6%**, ya que estás métricas si evalúan el orden en que se muestran los documentos, algo muy relevante para el usuario, que espera una mejor coincidencia en el primer resultado

## Entregables — Lab 3
- [X] `idf_bm25`, `bm25`, `buscar_bm25` desde cero + comparación y barrido (Parte A).
- [X] `qrels` de 5 consultas + las 5 métricas desde cero (Parte B).
- [X] Tabla comparativa TF-IDF vs BM25 y **decisión justificada con una métrica**.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
